<a href="https://colab.research.google.com/github/arundhatimsb/assignment/blob/main/sentence_embeddings_sa_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install torch matplotlib scikit-learn gdown sentence-transformers numpy scikit-learn --quiet

In [11]:
import gdown
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize

In [12]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

folder_id = '1oEYY-Io_8XUBQZXnGftNjseU46Yxc5FK'
gdown.download_folder(
    id=folder_id,
    output='dataset',
    quiet=True,
    use_cookies=False
)

DATA_DIR = 'dataset'

df_train_sa = pd.read_csv(f'{DATA_DIR}/train_sa_10000.csv')
df_train_en = pd.read_csv(f'{DATA_DIR}/train_en_10000.csv')
df_dev_sa   = pd.read_csv(f'{DATA_DIR}/dev_sa_1000.csv')
df_dev_en   = pd.read_csv(f'{DATA_DIR}/dev_en_1000.csv')
df_test_sa  = pd.read_csv(f'{DATA_DIR}/test_sa_1000.csv')
df_test_en  = pd.read_csv(f'{DATA_DIR}/test_en_1000.csv')

print(df_test_sa)

     Source_id                                        Sentence_sa
0            1  एक्लिप्स् इति प्रोग्रामर् कृते दोषान्वेषणे अपि...
1            2  विश्वासकारणादेव समभाषि मया वचः। इति यथा शास्त्...
2            3  तदा, तत्स्वयं ड्रैवर निमित्तम् अन्वेष्यति। अहं...
3            4  सर्वेभ्यः इटरेशन्-अर्थम्, iterator इतीदं प्रत्...
4            5  अपरं द्वितीयमुद्रायां तेन मोचितायां द्वितीयस्य...
..         ...                                                ...
995        996         रुग्णः रोग हर्तारौ वैद्यौ धन्यवादं करोति ।
996        997  तदा निथनेल् कथितवान् नासरन्नगरात किं कश्चिदुत्...
997        998  एवं, साधनपेटिकायाः उपरि कर्षयामि । इदमत्र अस्ति ।
998        999  रिसोर्स् इत्यस्य edit, hide, duplicate तथा del...
999       1000  ऽ ऽ ऽ शरीरं किञ्चित प्रसारयेत तथा शनै: नेत्रम ...

[1000 rows x 2 columns]


In [13]:
def merge_dataset(df_sa: pd.DataFrame,df_en: pd.DataFrame) -> pd.DataFrame:
  df = pd.merge(df_sa, df_en, on='Source_id', how='inner')
  df = df.dropna(subset=['Sentence_sa', 'Sentence_en']).reset_index(drop=True)
  print(f"{len(df)} aligned pairs loaded.")
  return df;

df_train = merge_dataset(df_train_sa,df_train_en)
df_test = merge_dataset(df_test_sa, df_test_en)
df_dev = merge_dataset(df_dev_sa, df_dev_en)

df_all = pd.concat([df_train, df_dev, df_test], ignore_index=True)
print(f"\nTotal pairs across all splits: {len(df_all)}")
df_all.head(3)


10000 aligned pairs loaded.
1000 aligned pairs loaded.
1000 aligned pairs loaded.

Total pairs across all splits: 12000


,Source_id,Sentence_sa,Sentence_en
0,1,"""Ctrl, S नुत्वा रक्षन्तु।""","Save it with Ctrl, S."
1,2,गुरुः छात्रान् एकवारं पाठयति ।,Teacher will teach the students only once.
2,3,चित्रचालनमिदं पुनः कर्तुं मया अस्याः राशेः चित...,"To recreate this animation, I have to take two..."


In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = 'sentence-transformers/LaBSE'
model = SentenceTransformer(model_name, device=device)
print(f"Model loaded. Native embedding dimension: {model.get_sentence_embedding_dimension()}")

model.safetensors:   0%|          | 0.00/1.88G [00:00<?, ?B/s]

In [1]:
def encode_sentences(sentences, model, batch_size=64, desc='Encoding'):
  embeddings = model.encode(
      sentences,
      batch_size = batch_size,
      show_progress_bar=True,
      convert_to_numpy=True,
      normalize_embeddings=True,
      device = device
  )
  return embeddings
print("Encoding sanskrit sentences.")
sa_sentences = df_all['Sentence_sa'].tolist()
emb_sa = encode_sentences(sa_sentences,model, desc='Sanskrit')

print("Encoding emglish sentences.")
en_sentences = df_all['Sentence_en'].tolist()
emb_en = encode_sentences(en_sentences,model, desc='English')
print(f"Raw Sanskrit embedding shape : {emb_sa.shape}")
print(f"Raw English  embedding shape : {emb_en.shape}")

In [9]:
target_dimension = 128
combined_emebedings = np.vstack([emb_sa, emb_en])
pca = PCA(n_components = target_dimension, random_state=seed)
pca.fit(combined_emebedings)

explained = pca.explained_variance_ratio_.sum() * 100
print(f"Cumulative explained variance in {target_dimension} components: {explained:.2f}%")

emb_sa_norm = normalize(pca.transform(emb_sa), norm='l2')
emb_en_norm = normalize(pca.transform(emb_en), norm='l2')

print(f"\n{'='*50}")
print(f"  EMBEDDING DIMENSION USED : {TARGET_DIM}")
print(f"{'='*50}")
print(f"Sanskrit embedding matrix shape : {emb_sa_norm.shape}")
print(f"English  embedding matrix shape : {emb_sa_norm.shape}")

NameError: name 'np' is not defined